In [1]:
import os

from IPython.display import FileLink

if os.getcwd() == '/notebooks':
    os.chdir("./motion-synthesis")
    print('inside dir: ', os.listdir())
print(os.listdir())
print("Click here to download the dit_d0.tar: ", display(FileLink("/notebooks/motion-synthesis/dit_d0.tar")))


os.environ["TOKENIZERS_PARALLELISM"] = "false"

inside dir:  ['requirements.txt', 'networks', 'dit_stable_crossattn_debug.tar', 'glove', 'exp_results', 'README.md', 'conda_requirements.txt', 'prepare', 'data_utils', 'dataset_split_builder.py', 'README_v1.md', 'demo.py', 'checkpoints', 'environment.yaml', 'dit_stable_crossattn_full.tar', 'eval_models', 'environment_cpu.yaml', '.git', 'assets', 'options', 'main.ipynb', 'motion-dataset.zip', 'common', 'lambda_requirements.txt', 'main.py', '.ipynb_checkpoints', 'data', '.gitignore', 'log', 'utils']
['requirements.txt', 'networks', 'dit_stable_crossattn_debug.tar', 'glove', 'exp_results', 'README.md', 'conda_requirements.txt', 'prepare', 'data_utils', 'dataset_split_builder.py', 'README_v1.md', 'demo.py', 'checkpoints', 'environment.yaml', 'dit_stable_crossattn_full.tar', 'eval_models', 'environment_cpu.yaml', '.git', 'assets', 'options', 'main.ipynb', 'motion-dataset.zip', 'common', 'lambda_requirements.txt', 'main.py', '.ipynb_checkpoints', 'data', '.gitignore', 'log', 'utils']


/notebooks/motion-synthesis/dit_d0.tar

Click here to download the dit_d0.tar:  None


In [2]:
import torch
from torch.backends import cuda
from options.train_options  import TrainOptions
from os.path import join as pjoin
import os
from utils.paramUtils import t2m_kinematic_chain
import numpy as np
from utils.word_vectorizer import WordVectorizer
from torch.utils.data import DataLoader
from data_utils.dataset import MotionDatasetV2
from data_utils.dataset import PartMotionDatasetV2
from networks.nn import MotionVQVAE, DiT, MotionVAE
from networks.trainers import MotionVQVAETrainer, MotionDiTTrainer, MotionVAETrainer
from torch.utils.data import Subset
from networks.nn_validator import VQVAEValidator, DiffusionValidator
import json

In [ ]:
parser = TrainOptions()
options = parser.parse(args = ['--stage', 'generation', '--max_epoch', '5', '--lr', '1e-4', '--save_latest', '5', '--eval_every_e', '1', '--save_every_e', '50', '--log_every', '1'])
options.gpu_id = torch.cuda.current_device() if torch.cuda.is_available() else -1
options.device = torch.device("cpu" if options.gpu_id==-1 else "cuda:" + str(options.gpu_id))
torch.autograd.set_detect_anomaly(True)

# disabling flash backend till the architecture parameters allow stable use of flash backend for attention
# without creating nans
cuda.enable_flash_sdp(False)
cuda.enable_mem_efficient_sdp(False)
cuda.enable_math_sdp(True)

if options.gpu_id != -1:
    # self.opt.gpu_id = int(self.opt.gpu_id)
    torch.cuda.set_device(options.gpu_id)

print('\nDevice used: ', options.device)
options.save_root = pjoin(options.checkpoints_dir, 'HumanML3D', options.name)
options.model_dir = pjoin(options.checkpoints_dir, 'model')
options.meta_dir = pjoin(options.save_root, 'meta')
options.eval_dir = pjoin(options.save_root, 'animation')
options.log_dir = pjoin('./log', options.dataset_name, options.name)
#options.experiment_dir = './exp_results/vae-setup/finetune_vae'
options.experiment_dir = './exp_results/stable-diffusion-setup/finetune_dit_cross_attn'
options.output_dir = options.experiment_dir
options.is_train = True
options.is_continue = False
options.dataset_mode = "debug"
options.batch_size = 64
options.model_filename = 'dit_stable_crossattn_nano.tar'

os.makedirs(options.model_dir, exist_ok=True)
os.makedirs(options.meta_dir, exist_ok=True)
os.makedirs(options.eval_dir, exist_ok=True)
os.makedirs(options.log_dir, exist_ok=True)

options.data_root = './data/HumanML3D'
options.motion_dir = pjoin(options.data_root, 'new_joint_vecs')
options.text_dir = pjoin(options.data_root, 'texts')
options.joints_num = 22
options.max_motion_length = 120
dim_pose = 263
radius = 4
fps = 20
kinematic_chain = t2m_kinematic_chain

print('Set parameters')
print('Learning rate: ', options.lr)
print('Max epochs: ', options.max_epoch)
print('Save latest frequency: ', options.save_latest)
print('Save every epoch frequency: ', options.save_every_e)
print('Log every iterations frequency: ', options.log_every)


Device used:  cuda:0
Set parameters
Learning rate:  0.0001
Max epochs:  5
Save latest frequency:  5
Save every epoch frequency:  50
Log every iterations frequency:  1


In [4]:
mean = np.load(pjoin(options.data_root, 'Mean.npy'))
std = np.load(pjoin(options.data_root, 'Std.npy'))

w_vectorizer = WordVectorizer('./glove', 'our_vab')
train_split_fn = 'train.txt'
val_split_fn = 'val.txt'
test_split_fn = 'test.txt'
simple_test_split_fn = 'simple_test.txt'

if options.dataset_mode in ["debug", "nano", "micro"]:
    train_split_fn = f'train_{options.dataset_mode}.txt'
    val_split_fn = f'val_{options.dataset_mode}.txt'

train_split_file = pjoin(options.data_root, train_split_fn)
val_split_file = pjoin(options.data_root, val_split_fn)
test_split_file = pjoin(options.data_root, test_split_fn)
simple_test_split_file = pjoin(options.data_root, simple_test_split_fn)

train_dataset = PartMotionDatasetV2(options, mean, std, train_split_file)
val_dataset = PartMotionDatasetV2(options, mean, std, val_split_file)
test_dataset = PartMotionDatasetV2(options, mean, std, test_split_file)

print('\nTotal number of snippets in train: ', len(train_dataset))
print('Total number of snippets in val: ', len(val_dataset))
print('Total number of snippets in test: ', len(test_dataset))

sample_motion = train_dataset[4]
print('Sample data shape: ', sample_motion['motion_parts'].shape, sample_motion['text'])
Dp_max = sample_motion['motion_parts'].shape[-1]

id list 128


100%|██████████| 128/128 [00:01<00:00, 78.74it/s]


Motion shape (B, T, D): (382, 120, 263)
Total number of motions 382
Total number of small motions: 61
id list 64


100%|██████████| 64/64 [00:00<00:00, 113.44it/s]


Motion shape (B, T, D): (191, 120, 263)
Total number of motions 191
Total number of small motions: 26
id list 4384


100%|██████████| 4384/4384 [00:34<00:00, 127.75it/s]

Motion shape (B, T, D): (13114, 120, 263)
Total number of motions 13114
Total number of small motions: 1710

Total number of snippets in train:  382
Total number of snippets in val:  191
Total number of snippets in test:  13114
Sample data shape:  (120, 6, 60) the person is jumping up and down.


In [5]:
train_loader = DataLoader(train_dataset, batch_size=options.batch_size, drop_last=not(options.dataset_mode in ['micro', 'nano']), num_workers=1,
                              shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=options.batch_size, drop_last=not(options.dataset_mode in ['micro', 'nano']), num_workers=1,
                        shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=options.batch_size, drop_last=not(options.dataset_mode in ['micro', 'nano']), num_workers=1,
                         shuffle=True, pin_memory=True)

if options.stage == "autoencoder":
    '''
    vqvae = MotionVQVAE(
        input_dim=Dp_max,
        enc_hidden_dim=1024,
        dec_hidden_dim=1024,
        latent_dim=256,
        num_embeddings=512,
        beta=0.1
    )

    if options.is_train:
        trainer = MotionVQVAETrainer(options, vqvae = vqvae)
        trainer.train(
            train_dataloader=train_loader,
            val_dataloader=val_loader)
    '''
    vae = MotionVAE(
        dim = 263, #input dimension of motion vector
        hidden_size = 512, # latent dimension
        max_seq_len=options.max_motion_length,
        num_heads = 4,
        depth = 9
    )
    if options.is_train:
        trainer = MotionVAETrainer(options, vae = vae)
        trainer.train(
            train_dataloader=train_loader,
            val_dataloader=val_loader
        )
else:
    dit = DiT(
        input_size = 512, # latent dimension
        hidden_size = 1152,
        text_dim = 768,
        max_seq_len=options.max_motion_length // 4
    )

    if options.is_train:
        trainer = MotionDiTTrainer(
            args = options,
            dit = dit,
            autoencoder_type="pretrained_vae"
        )
        trainer.train(
            train_dataloader=train_loader,
            val_dataloader=val_loader
        )


tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Number of epochs: 5
Number of steps: 30
Iters Per Epoch, Training: 0006, Validation: 003

a man is sweeping the floor.
the person recovered from falling.
a person who seems to sit
the man shrugs raising his left side a little higher than the right.
person is doing situps.
the figure sprints forward a short distance before crouching down and touching the ground with both hands.
a person bows forward then steps backward.
a person raises their right hand and revolves it in front of them, and then raises both their arms above their head.
a person does a squat while slightly raising hands
Epoch: 0
Train Loss: 0.84678
Validation Loss: 0.86398
he starts washing his head
a person stretches their arms, moving the shoulders, elbows then wrists.
subject is stretching with left arm, then the right arm.
a person walks forward slowly.
the person is sipping  a martini to drink.
walking forward, turning around on one foot then walking back
a person performing a swift strafing movement to the right
a p

In [6]:
if options.stage == "autoencoder":
    test_model_filepath = pjoin(options.model_dir, options.model_filename)

    if not(options.is_train) and os.path.exists(test_model_filepath):
        vae_model_dict = torch.load(test_model_filepath, map_location = options.device)

        vae.load_state_dict(vae_model_dict['vae'])

        vae_validator = VAEValidator(
            opt = options,
            vqvae=vae,
            train_dataloader=train_loader,
            val_dataloader = val_loader
        )
        vqvae_validator.validate()
    else:
        print("Invalid mode or model file doesn't exist!")
else:
    test_model_filepath = pjoin(options.model_dir, options.model_filename)
    if not(options.is_train) and os.path.exists(test_model_filepath):
        dit_model_dict = torch.load(test_model_filepath, map_location = options.device)

        dit.load_state_dict(dit_model_dict['dit'])

        dit_validator = DiffusionValidator(
            opt = options,
            dit=dit,
            val_dataloader = val_loader,
            test_dataloader=test_loader,
            test_type = "test_simple"
        )
        dit_validator.validate()
    else:
        print("Invalid mode or model file doesn't exist!")

Invalid mode or model file doesn't exist!
